# IDLE Detector — Analysis and Validation

Analysis of a threshold-based IDLE detector using CSI amplitude and phase metrics.

**Proposed pipeline:**
```
H_raw(t) → [metrics] → [adaptive threshold] → IDLE / NO-IDLE
                                                    │
                                IDLE   → update H_s (EMA)
                                NO-IDLE → H_d = H - H_s → HAR classifier
```

**Metrics used:**
- `amp_var`: temporal variance of |H| averaged over subcarriers
- `phase_diff_var`: variance of frame-to-frame phase differences
- `combined`: sum of the two

In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import roc_curve, roc_auc_score

# importa le funzioni da idle_detector.py
sys.path.insert(0, '.')
from idle_detector import (
    motion_metric, amp_var_only, phase_diff_var_only,
    IdleDetector, load_dataset,
    CLASSES, CLASS_COLORS, AMP_IDX, PHASE_IDX
)

%matplotlib inline
plt.rcParams.update({
    'font.size': 12, 'axes.titlesize': 13, 'axes.labelsize': 12,
    'legend.fontsize': 10, 'figure.dpi': 120,
    'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
})

In [2]:
X_sim,  y_sim  = load_dataset('dataset_v2/dataset_har_combined.npz')
X_real, y_real = load_dataset('dataset_real_v2/dataset_har_combined.npz')

# ricostruisce canale complesso dalla shape (N, T, N_SC, 4)
# feature 0=re, 1=im, 2=|H|, 3=phase
H_sim  = (X_sim[..., 0]  + 1j * X_sim[..., 1]).astype(np.complex64)
H_real = (X_real[..., 0] + 1j * X_real[..., 1]).astype(np.complex64)
print(f'H_sim  : {H_sim.shape}   H_real: {H_real.shape}')

Loading dataset_v2/dataset_har_combined.npz ...


  shape=(1200, 40, 1272, 4)  classi=[np.str_('FALL'), np.str_('IDLE'), np.str_('JUMP'), np.str_('RUN'), np.str_('STAND'), np.str_('WALK')]
Loading dataset_real_v2/dataset_har_combined.npz ...


  shape=(300, 40, 1272, 4)  classi=[np.str_('FALL'), np.str_('IDLE'), np.str_('JUMP'), np.str_('RUN'), np.str_('STAND'), np.str_('WALK')]


H_sim  : (1200, 40, 1272)   H_real: (300, 40, 1272)


## 1. What the detector sees — H(t) traces per class

Raw H(t) amplitude and phase for IDLE vs. activity classes. This motivates why temporal variance works as a discriminant.

In [ ]:
SHOW_CLASSES = ['IDLE', 'STAND', 'WALK', 'FALL']
SC_STRIDE = 16
SC_SHOW   = np.arange(0, 1272, SC_STRIDE)

fig, axes = plt.subplots(len(SHOW_CLASSES), 2, figsize=(16, 3.5 * len(SHOW_CLASSES)))
fig.suptitle('Raw H(t) — amplitude and phase — one sample per class (real data)', fontsize=14)

for row, cls in enumerate(SHOW_CLASSES):
    idx = np.where(y_real == cls)[0][0]
    H_ex = H_real[idx]
    amp_ex   = np.abs(H_ex)[:, SC_SHOW]
    phase_ex = np.angle(H_ex)[:, SC_SHOW]

    for col, (data, label) in enumerate([(amp_ex, '|H|'), (phase_ex, 'angle(H) [rad]')]):
        ax = axes[row, col]
        im = ax.imshow(data.T, aspect='auto', origin='lower',
                       cmap='viridis' if col == 0 else 'RdBu',
                       interpolation='nearest')
        plt.colorbar(im, ax=ax, shrink=0.8)
        ax.set_xlabel('Time frame (t)')
        ax.set_ylabel('Subcarrier (downsampled)')
        ax.set_title(f'[{cls}] {label}')

plt.tight_layout()
plt.show()

## 2. Scalar metrics — per-class distribution

Each sample (N, T, N_SC, 4) is reduced to a single scalar. A good IDLE detector requires IDLE to have systematically the lowest metric value across all classes.

In [ ]:
metrics_def = {
    'Combined\n(amp_var + phase_diff_var)': motion_metric,
    'Amplitude\ntemporal variance':         amp_var_only,
    'Phase\nframe-to-frame var':            phase_diff_var_only,
}

fig, axes = plt.subplots(len(metrics_def), 2, figsize=(14, 4 * len(metrics_def)))
fig.suptitle('Metric distribution per class — IDLE should be the lowest', fontsize=13)

for row, (label, fn) in enumerate(metrics_def.items()):
    for col, (X, y, domain) in enumerate([
        (X_sim,  y_sim,  'Simulated'),
        (X_real, y_real, 'Real'),
    ]):
        ax = axes[row, col]
        data   = [fn(X[y == cls]) for cls in CLASSES]
        colors = [CLASS_COLORS[cls] for cls in CLASSES]
        bp = ax.boxplot(data, patch_artist=True, tick_labels=CLASSES,
                        medianprops=dict(color='black', linewidth=2),
                        flierprops=dict(marker='o', markersize=3, alpha=0.4))
        for patch, c in zip(bp['boxes'], colors):
            patch.set_facecolor(c)
            patch.set_alpha(0.75)
        ax.set_yscale('log')
        ax.set_ylabel(label if col == 0 else '')
        ax.set_title(domain if row == 0 else '')

        idle_median = np.median(fn(X[y == 'IDLE']))
        ax.axhline(idle_median, color='steelblue', lw=1.5,
                   linestyle='--', alpha=0.8, label='IDLE median')
        if row == 0 and col == 1:
            ax.legend()

plt.tight_layout()
plt.show()

## 3. 2D metric space — amp_var vs phase_diff_var scatter

Shows whether the two metrics are complementary or redundant, and where IDLE clusters relative to other classes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('2D metric space — each point is one sample', fontsize=13)

for ax, (X, y, domain) in zip(axes, [
    (X_sim,  y_sim,  'Simulated'),
    (X_real, y_real, 'Real'),
]):
    for cls in CLASSES:
        mask = y == cls
        av = amp_var_only(X[mask])
        pv = phase_diff_var_only(X[mask])
        ax.scatter(av, pv, c=CLASS_COLORS[cls], label=cls,
                   alpha=0.6, s=20, edgecolors='none')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('amp_var  (temporal variance of |H|)')
    ax.set_ylabel('phase_diff_var  (variance of frame-to-frame Δφ)')
    ax.set_title(domain)
    ax.legend(markerscale=2, ncol=2)

plt.tight_layout()
plt.show()

## 4. ROC — how well IDLE is separated from all other activities

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('ROC — IDLE=0 vs. any activity=1', fontsize=13)

metric_fns = {
    'Combined':   motion_metric,
    'Amplitude':  amp_var_only,
    'Phase diff': phase_diff_var_only,
}

for ax, (X, y, domain) in zip(axes, [
    (X_sim,  y_sim,  'Simulated'),
    (X_real, y_real, 'Real'),
]):
    y_bin = (y != 'IDLE').astype(int)
    for label, fn in metric_fns.items():
        scores = fn(X)
        fpr, tpr, _ = roc_curve(y_bin, scores)
        auc = roc_auc_score(y_bin, scores)
        ax.plot(fpr, tpr, lw=2, label=f'{label}  AUC={auc:.3f}')

    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('FPR (false alarms — activity classified as IDLE)')
    ax.set_ylabel('TPR (activity correctly detected)')
    ax.set_title(domain)
    ax.legend()

plt.tight_layout()
plt.show()

## 5. ROC per individual class — IDLE vs. each activity separately

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ROC per class — combined metric — IDLE vs. each activity', fontsize=13)

NON_IDLE = [c for c in CLASSES if c != 'IDLE']

for ax, (X, y, domain) in zip(axes, [
    (X_sim,  y_sim,  'Simulated'),
    (X_real, y_real, 'Real'),
]):
    idle_scores = motion_metric(X[y == 'IDLE'])
    for cls in NON_IDLE:
        cls_scores = motion_metric(X[y == cls])
        scores = np.concatenate([idle_scores, cls_scores])
        labels = np.concatenate([np.zeros(len(idle_scores)), np.ones(len(cls_scores))])
        fpr, tpr, _ = roc_curve(labels, scores)
        auc = roc_auc_score(labels, scores)
        ax.plot(fpr, tpr, lw=2, color=CLASS_COLORS[cls],
                label=f'{cls}  AUC={auc:.3f}')

    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(domain)
    ax.legend()

plt.tight_layout()
plt.show()

## 6. Real-time pipeline simulation

Synthetic sequence: N IDLE samples (cold-start warmup) → N activity samples.  
The `IdleDetector` updates the adaptive threshold and shows where it makes errors (green = correct, red = wrong).

In [ ]:
def run_pipeline_demo(X, y, domain, act_list, n_idle=15, n_act=25,
                      n_init=5, k_sigma=3.0, ema_alpha=0.95):
    fig, axes = plt.subplots(1, len(act_list),
                             figsize=(6 * len(act_list), 5), sharey=False)
    fig.suptitle(f'Real-time pipeline — {domain} — green=IDLE correct, red=wrong', fontsize=13)

    if len(act_list) == 1:
        axes = [axes]

    for ax, act in zip(axes, act_list):
        idle_pool = X[y == 'IDLE']
        act_pool  = X[y == act]
        seq = np.concatenate([idle_pool[:n_idle], act_pool[:n_act]])
        gt  = ['IDLE'] * n_idle + [act] * n_act

        det = IdleDetector(n_init=n_init, k_sigma=k_sigma, ema_alpha=ema_alpha)
        metric_vals, thresholds, preds = [], [], []
        for X_win in seq:
            is_idle, _ = det.update(X_win)
            metric_vals.append(det._metric(X_win))
            thresholds.append(det.threshold or np.nan)
            preds.append(is_idle)

        ax.semilogy(np.arange(len(seq)), metric_vals, color='steelblue', lw=1.8, label='metric')
        ax.semilogy(np.arange(len(seq)), thresholds,  color='red', lw=1.5,
                    linestyle='--', label='adaptive threshold')
        for i, (p, g) in enumerate(zip(preds, gt)):
            correct = (p and g == 'IDLE') or (not p and g != 'IDLE')
            ax.axvspan(i - 0.5, i + 0.5, alpha=0.15,
                       color='#2ecc71' if correct else '#e74c3c')
        ax.axvline(n_idle - 0.5, color='black', lw=2, linestyle=':')
        ax.text(n_idle + 0.3, np.nanmax(metric_vals) * 0.5, f'← {act}', fontsize=10)
        acc = np.mean([(p and g == 'IDLE') or (not p and g != 'IDLE')
                       for p, g in zip(preds, gt)]) * 100
        ax.set_title(f'{act}   acc={acc:.0f}%')
        ax.set_xlabel('Sample index')
        ax.set_ylabel('Metric (log scale)')
        ax.legend()

    plt.tight_layout()
    plt.show()


print('=== REAL dataset ===')
run_pipeline_demo(X_real, y_real, 'Real',
                  act_list=['WALK', 'RUN', 'FALL', 'STAND'],
                  n_idle=15, n_act=25)

In [ ]:
print('=== SIMULATED dataset ===')
run_pipeline_demo(X_sim, y_sim, 'Simulated',
                  act_list=['WALK', 'RUN', 'FALL', 'STAND'],
                  n_idle=15, n_act=25)

## 7. Effect of k_sigma on the threshold

`k_sigma` controls how conservative the threshold is:
- **small k** → lower threshold → more false alarms (activity classified IDLE → H_s contaminated)
- **large k** → higher threshold → more miss detections (IDLE classified as activity → H_s not updated)

In [ ]:
K_VALUES = [1.5, 2.0, 3.0, 5.0]
ACT_TEST = 'WALK'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Effect of k_sigma — sequence IDLE→{ACT_TEST} — real dataset', fontsize=13)

idle_pool = X_real[y_real == 'IDLE']
act_pool  = X_real[y_real == ACT_TEST]
seq = np.concatenate([idle_pool[:15], act_pool[:25]])
gt  = ['IDLE'] * 15 + [ACT_TEST] * 25

det0 = IdleDetector(n_init=5, k_sigma=3.0)
metrics_gt = []
for X_win in seq:
    det0.update(X_win)
    metrics_gt.append(det0._metric(X_win))

ax_acc, ax_thr = axes
accs = []
for k in K_VALUES:
    det = IdleDetector(n_init=5, k_sigma=k)
    preds, thrs = [], []
    for X_win in seq:
        is_idle, _ = det.update(X_win)
        preds.append(is_idle)
        thrs.append(det.threshold or np.nan)
    acc = np.mean([(p and g == 'IDLE') or (not p and g != 'IDLE')
                   for p, g in zip(preds, gt)]) * 100
    accs.append(acc)
    ax_thr.semilogy(np.arange(len(seq)), thrs, lw=1.8, label=f'k={k}  acc={acc:.0f}%')

ax_thr.semilogy(np.arange(len(seq)), metrics_gt,
                color='black', lw=1, linestyle=':', label='metric')
ax_thr.axvline(14.5, color='black', lw=1.5, linestyle='--')
ax_thr.set_title('Adaptive threshold vs. k')
ax_thr.set_xlabel('Sample index'); ax_thr.set_ylabel('Value (log)')
ax_thr.legend()

ax_acc.bar([str(k) for k in K_VALUES], accs,
           color=['#3498db', '#2ecc71', '#e67e22', '#e74c3c'])
ax_acc.set_xlabel('k_sigma')
ax_acc.set_ylabel('Accuracy (%)')
ax_acc.set_title(f'IDLE detector accuracy vs. k — IDLE→{ACT_TEST}')
ax_acc.set_ylim(0, 105)
for i, v in enumerate(accs):
    ax_acc.text(i, v + 1, f'{v:.0f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## 8. H_s estimation quality

During IDLE the detector accumulates H_s via EMA. Verify that H_s converges to the real static channel and that H_d = H - H_s makes sense during activity.

In [ ]:
ACT_DEMO = 'WALK'
N_IDLE_DEMO, N_ACT_DEMO = 20, 20

idle_pool = X_real[y_real == 'IDLE']
act_pool  = X_real[y_real == ACT_DEMO]
seq = np.concatenate([idle_pool[:N_IDLE_DEMO], act_pool[:N_ACT_DEMO]])

det = IdleDetector(n_init=5, k_sigma=3.0, ema_alpha=0.95)
H_s_history, H_d_frames, is_idle_seq = [], [], []

for X_win in seq:
    is_idle, H_s = det.update(X_win)
    is_idle_seq.append(is_idle)
    if H_s is not None:
        H_s_history.append(H_s.copy())
        H_amp_mean = X_win[..., AMP_IDX].mean(axis=0)
        H_d_frames.append(np.abs(H_amp_mean - H_s))

H_s_hist = np.array(H_s_history)
H_d_hist  = np.array(H_d_frames)
SC_STRIDE2 = 8

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle(f'H_s and H_d evolution — IDLE×{N_IDLE_DEMO} → {ACT_DEMO}×{N_ACT_DEMO} (real data)', fontsize=13)

im0 = axes[0].imshow(H_s_hist[:, ::SC_STRIDE2].T, aspect='auto', origin='lower', cmap='viridis')
axes[0].axvline(N_IDLE_DEMO - 0.5, color='red', lw=2, linestyle='--')
axes[0].set_title('Estimated H_s over time (converges during IDLE, then frozen)')
axes[0].set_ylabel('Subcarrier')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(H_d_hist[:, ::SC_STRIDE2].T, aspect='auto', origin='lower', cmap='hot')
axes[1].axvline(N_IDLE_DEMO - 0.5, color='cyan', lw=2, linestyle='--')
axes[1].set_title('H_d = |H| - H_s  (near zero during IDLE, non-zero during activity)')
axes[1].set_ylabel('Subcarrier')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

hd_energy = (H_d_hist**2).mean(axis=1)
axes[2].semilogy(hd_energy, color='darkorange', lw=2)
axes[2].axvline(N_IDLE_DEMO - 0.5, color='red', lw=2, linestyle='--', label=f'{ACT_DEMO} start')
for i, p in enumerate(is_idle_seq[:len(hd_energy)]):
    axes[2].axvspan(i - 0.5, i + 0.5, alpha=0.1, color='#2ecc71' if p else '#e74c3c')
axes[2].set_title('H_d energy per sample (green=IDLE detected, red=activity detected)')
axes[2].set_xlabel('Sample index')
axes[2].set_ylabel('Mean ||H_d||²')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 9. Figure per il paper (publication quality)

Genera le due figure da includere nel paper in `figs/idle_detector/paper/`.
Stile serif, scala log, inglese. Quando pronte si spostano in `paper-src/...`.

In [ ]:
import os, shutil
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from idle_detector import motion_metric

matplotlib.rcParams.update({
    'font.size':        11,
    'axes.titlesize':   11,
    'axes.labelsize':   10,
    'xtick.labelsize':  10,
    'ytick.labelsize':   9,
    'legend.fontsize':  10,
})

CLASSES = ['IDLE', 'STAND', 'WALK', 'RUN', 'FALL', 'JUMP']
COLORS  = {
    'IDLE':  '#3498db', 'STAND': '#95a5a6', 'WALK': '#f39c12',
    'RUN':   '#e67e22', 'FALL':  '#e74c3c', 'JUMP': '#1abc9c',
}

M_all  = motion_metric(X_real)
M_idle = M_all[y_real == 'IDLE']
thr    = np.mean(M_idle) + 3.0 * np.std(M_idle)
data   = [M_all[y_real == cls] for cls in CLASSES]

fig, ax = plt.subplots(figsize=(7.0, 4.2))
fig.subplots_adjust(left=0.12, right=0.97, top=0.93, bottom=0.12)

parts = ax.violinplot(data, positions=range(len(CLASSES)),
                      showmedians=True, showextrema=False, widths=0.7)

for pc, cls in zip(parts['bodies'], CLASSES):
    pc.set_facecolor(COLORS[cls])
    pc.set_edgecolor(COLORS[cls])
    pc.set_alpha(0.55)

parts['cmedians'].set_color('black')
parts['cmedians'].set_linewidth(1.8)

ax.axhline(thr, color='#2c3e50', lw=1.8, linestyle='--',
           label=f'Detection threshold  $\\mu_0 + 3\\,\\sigma_0$')
ax.axhspan(0, thr, alpha=0.06, color='#3498db')

ax.set_yscale('log')
ax.set_xticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES)
ax.set_ylabel('Combined metric $\\mathcal{M}$  (log scale)')
ax.set_title('Motion metric distribution per activity — real measurements')
ax.legend(loc='upper left', framealpha=0.9)
ax.grid(axis='y', alpha=0.25, linestyle=':')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

all_vals = np.concatenate(data)
ax.set_ylim(all_vals.min() * 0.4, all_vals.max() * 3.0)

PAPER_OUT = 'figs/idle_detector/paper'
PAPER_DST = 'paper-src/CNIT-2026-technical-report-chapter/figures/idle_detector'
os.makedirs(PAPER_OUT, exist_ok=True)
os.makedirs(PAPER_DST, exist_ok=True)
for ext in ('pdf', 'png'):
    src = f'{PAPER_OUT}/idle_metric_dist.{ext}'
    dst = f'{PAPER_DST}/idle_metric_dist.{ext}'
    fig.savefig(src, bbox_inches='tight', dpi=200)
    shutil.copy2(src, dst)
    print(f'Saved: {src}  ->  {dst}')

plt.show()
plt.close(fig)


### 9b. Paper figure — Real-time pipeline simulation

2 rows (Simulated / Real) x 3 columns (WALK / RUN / FALL).  
Saves to `figs/idle_detector/paper/idle_pipeline_realtime.pdf/.png`,  
then copies to `paper-src/.../figures/idle_detector/`.

In [ ]:
import os, shutil
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from idle_detector import IdleDetector

# default matplotlib font, just tweak sizes
matplotlib.rcParams.update({
    'font.size':       10,
    'axes.titlesize':  10,
    'axes.labelsize':   9,
    'xtick.labelsize':  8,
    'ytick.labelsize':  8,
    'lines.linewidth':  1.6,
})

ACTS   = ['WALK', 'RUN', 'FALL']
N_IDLE, N_ACT, N_INIT, K = 15, 25, 5, 3.0

# run detector on real data for each activity
results = {}
for act in ACTS:
    idle_pool = X_real[y_real == 'IDLE']
    act_pool  = X_real[y_real == act]
    seq = np.concatenate([idle_pool[:N_IDLE], act_pool[:N_ACT]])
    gt  = ['IDLE'] * N_IDLE + [act] * N_ACT

    det = IdleDetector(n_init=N_INIT, k_sigma=K)
    mv, thr, preds = [], [], []
    for win in seq:
        is_idle, _ = det.update(win)
        mv.append(det._metric(win))
        thr.append(det.threshold if det.threshold else float('nan'))
        preds.append(is_idle)
    results[act] = dict(mv=mv, thr=thr, preds=preds, gt=gt)

# shared y-axis: tight bounds from actual data
all_mv = [v for act in ACTS for v in results[act]['mv'] if v > 0]
y_min  = min(all_mv) * 0.5
y_max  = max(all_mv) * 3.0

fig, axes = plt.subplots(1, 3, figsize=(11, 3.8), sharey=True)
fig.subplots_adjust(wspace=0.08, bottom=0.28)

for col, act in enumerate(ACTS):
    ax  = axes[col]
    r   = results[act]
    mv  = r['mv']; thr = r['thr']; preds = r['preds']; gt = r['gt']
    t   = list(range(len(mv)))

    # background shading: green = correct, red = wrong
    for i, (p, g) in enumerate(zip(preds, gt)):
        ok = (p and g == 'IDLE') or (not p and g != 'IDLE')
        ax.axvspan(i - 0.5, i + 0.5, alpha=0.13,
                   color='#27ae60' if ok else '#e74c3c', zorder=1)

    # IDLE/activity boundary (dotted vertical line)
    ax.axvline(N_IDLE - 0.5, color='#2c3e50', lw=1.3, linestyle=':', zorder=4)

    # metric and adaptive threshold
    l_met, = ax.semilogy(t, mv,  color='steelblue', lw=1.8, zorder=3)
    l_thr, = ax.semilogy(t, thr, color='#c0392b', lw=1.3, linestyle='--', zorder=3)

    ax.set_ylim(y_min, y_max)
    ax.set_xlim(-0.8, len(mv) - 0.2)

    acc = sum((p and g == 'IDLE') or (not p and g != 'IDLE')
              for p, g in zip(preds, gt)) / len(preds) * 100
    ax.set_title(f'{act}   (accuracy = {acc:.0f} %)', pad=5)
    ax.set_xlabel('Window index')
    if col == 0:
        ax.set_ylabel('Combined metric $\\mathcal{M}$ (log scale)')
    ax.grid(axis='y', alpha=0.22, linestyle=':')

# single shared legend below all panels
p_ok  = mpatches.Patch(color='#27ae60', alpha=0.45)
p_err = mpatches.Patch(color='#e74c3c', alpha=0.45)
l_thr_h = mlines.Line2D([], [], color='#c0392b', lw=1.3, linestyle='--')
fig.legend(
    handles=[l_met, l_thr_h, p_ok, p_err],
    labels=[
        'Motion metric $\\mathcal{M} = \\sigma^2_{|H|} + \\sigma^2_{\\Delta\\phi}$',
        'Adaptive threshold $\\mu_0 + k\\,\\sigma_0$  —  IDLE/active boundary',
        'Window correctly classified (IDLE or active)',
        'Window misclassified',
    ],
    loc='lower center',
    ncol=2,
    fontsize=8.5,
    framealpha=0.9,
    bbox_to_anchor=(0.5, 0.01),
)

PAPER_OUT = 'figs/idle_detector/paper'
PAPER_DST = 'paper-src/CNIT-2026-technical-report-chapter/figures/idle_detector'
os.makedirs(PAPER_OUT, exist_ok=True)
os.makedirs(PAPER_DST, exist_ok=True)
for ext in ('pdf', 'png'):
    src = f'{PAPER_OUT}/idle_pipeline_realtime.{ext}'
    dst = f'{PAPER_DST}/idle_pipeline_realtime.{ext}'
    fig.savefig(src, bbox_inches='tight', dpi=200)
    shutil.copy2(src, dst)
    print(f'Saved: {src}  ->  {dst}')

plt.show()
plt.close(fig)
